# Notebook 02 — Data Cleaning & Feature Engineering

**Units**: All parsed dimensions are in **hundredths of an inch** (0.01 inch),
which matches the `PRODUCT_LENGTH` target unit.
- 1 inch = 100 units
- 1 cm   = 39.37 units
- 1 mm   = 3.937 units
- 1 ft   = 1200 units

**Outputs** saved to `../processed_features/`:
- `X_train/val/test_features.parquet`
- `train_indices.npy`, `val_indices.npy`

All code self-contained.

## 0. DATA MODE

In [1]:
# ============================================================
# DATA MODE: Changed to experiment mode
# ============================================================
DATA_MODE = "experiment"
DATA_PATHS = {
    "debug":      "../dataset/sampled/debug",
    "experiment": "../dataset/sampled/experiment",
    "full":       "../dataset"
}
DATA_DIR = DATA_PATHS[DATA_MODE]
print("Using dataset:", DATA_DIR)


Using dataset: ../dataset/sampled/experiment


## 1. Imports

In [2]:
import os, re, numpy as np, pandas as pd
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', None)
OUTPUT_DIR = "../processed_features"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("✅ Imports done. Output dir:", os.path.abspath(OUTPUT_DIR))


✅ Imports done. Output dir: d:\AmazonML\processed_features


## 2. Text Cleaning

In [3]:
def clean_text(text):
    if not isinstance(text, str) or not text.strip():
        return ''
    text = text.lower().strip()
    text = re.sub(r'[\u00d7\u2715]', ' x ', text)   # × and ✕ → x
    text = re.sub(r'[^\x00-\x7f]', ' ', text)        # remove non-ASCII
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

assert clean_text("Bag 12×8×4 Inches") == "bag 12 x 8 x 4 inches"
print("✅ clean_text OK")


✅ clean_text OK


## 3. Dimension Parser

**All output values are in PRODUCT_LENGTH units (hundredths of an inch = 0.01 inch)**

Conversion table:
- 1 inch = 100 units  
- 1 cm   = 39.37 units  
- 1 mm   = 3.937 units  
- 1 ft   = 1200 units

In [4]:
# Units → PRODUCT_LENGTH units (1 unit = 0.01 inch)
UNIT_TO_PL = {
    'in':   100.0,  'inch':   100.0,  'inches':  100.0,  '"': 100.0,
    'ft':  1200.0,  'foot':  1200.0,  'feet':   1200.0,  "'":1200.0,
    'yd':  3600.0,  'yard':  3600.0,  'yards':  3600.0,
    'cm':    39.37, 'centimeter': 39.37, 'centimeters': 39.37,
    'mm':     3.937,'millimeter':  3.937,'millimeters':  3.937,
    'm':   3937.0,  'meter':  3937.0, 'meters':  3937.0
}

_UNITS = r'(cm|mm|m|in|inch|inches|ft|feet|foot|yd|yard|yards|["\'\u2033\u2032])'
_NUM   = r'(\d+(?:[.,]\d+)?)'

RE_MULTI  = re.compile(
    _NUM + r'\s*(?:x|×|X|by)\s*' + _NUM +
    r'(?:\s*(?:x|×|X|by)\s*' + _NUM + r')?\s*' + _UNITS, re.I)
RE_SINGLE = re.compile(_NUM + r'\s*' + _UNITS, re.I)
RE_EXPLEN = re.compile(
    r'(?:product\s+length|item\s+length|overall\s+length|length)\s*[:=\-]?\s*'
    + _NUM + r'\s*' + _UNITS
    + r'|' + _NUM + r'\s*' + _UNITS + r'\s+(?:in\s+)?length',
    re.I
)
RE_IGNORE = re.compile(r'\b(?:mah|gb|tb|w|hz|v|gsm|pcs|pack|set|dpi|mp|k|fps|g|kg|lbs|oz)\b', re.I)

def _conv(val_str, unit):
    """Convert to PRODUCT_LENGTH units (hundredths of an inch)."""
    try:
        v = float(str(val_str).replace(',', '.'))
        return v * UNIT_TO_PL.get(unit.lower().strip(), 0.0)
    except:
        return 0.0

def parse_dimensions(text):
    """Returns dict of features, all dim values in PRODUCT_LENGTH units (0.01 inch)."""
    out = dict(
        explicit_length_u=0.0,  has_explicit_length=0,
        dim_1_u=0.0, dim_2_u=0.0, dim_3_u=0.0,
        max_dim_u=0.0, min_dim_u=0.0, mid_dim_u=0.0,
        volume_u3=0.0, measurement_count=0, number_count=0,
        has_lxw=0, has_lxwxh=0,
        has_inch=0, has_cm=0, has_mm=0, has_ft=0
    )
    if not text or not isinstance(text, str):
        return out

    s = str(text)
    out['number_count'] = len(re.findall(r'\d+(?:[.,]\d+)?', s))

    # --- 1. Explicit "length: X unit" phrases ---
    for m in RE_EXPLEN.finditer(s):
        gs = [g for g in m.groups() if g]
        if len(gs) >= 2:
            nums  = [g for g in gs if re.match(r'^\d', g)]
            units = [g for g in gs if g.lower() in UNIT_TO_PL]
            if nums and units:
                v = _conv(nums[0], units[0])
                if 2 <= v <= 3_000_000:
                    out['explicit_length_u'] = v
                    out['has_explicit_length'] = 1
                    break

    # --- 2. L x W x H patterns ---
    dims = []
    for m in RE_MULTI.finditer(s):
        gs = m.groups()   # (d1, d2, d3_or_None, unit)
        unit = gs[-1]
        u = unit.lower() if unit else ''
        if 'in' in u or '"' in u: out['has_inch'] = 1
        elif 'cm' in u: out['has_cm'] = 1
        elif 'mm' in u: out['has_mm'] = 1
        elif 'ft' in u or 'foot' in u or 'feet' in u: out['has_ft'] = 1

        d1 = _conv(gs[0], unit) if gs[0] else 0.0
        d2 = _conv(gs[1], unit) if gs[1] else 0.0
        d3 = _conv(gs[2], unit) if len(gs) > 3 and gs[2] else 0.0
        for v in [d1, d2, d3]:
            if 2 <= v <= 3_000_000:
                dims.append(v)
        if d2 > 0: out['has_lxw'] = 1
        if d3 > 0: out['has_lxwxh'] = 1
        if dims:
            break

    # --- 3. Single dim fallback ---
    if not dims:
        for m in RE_SINGLE.finditer(s):
            val_s, unit_s = m.group(1), m.group(2)
            ctx = s[max(0, m.start()-10):m.end()+10]
            if RE_IGNORE.search(ctx):
                continue
            v = _conv(val_s, unit_s)
            if 2 <= v <= 3_000_000:
                u = unit_s.lower()
                if 'in' in u or '"' in u: out['has_inch'] = 1
                elif 'cm' in u: out['has_cm'] = 1
                elif 'mm' in u: out['has_mm'] = 1
                elif 'ft' in u: out['has_ft'] = 1
                dims.append(v)

    if dims:
        out['dim_1_u'] = dims[0]
        out['dim_2_u'] = dims[1] if len(dims) > 1 else 0.0
        out['dim_3_u'] = dims[2] if len(dims) > 2 else 0.0
        out['max_dim_u'] = max(dims)
        out['min_dim_u'] = min(dims)
        sd = sorted(dims)
        out['mid_dim_u'] = sd[len(sd)//2]
        out['measurement_count'] = len(dims)
        d1 = dims[0]
        d2 = dims[1] if len(dims) > 1 else 1.0
        d3 = dims[2] if len(dims) > 2 else 1.0
        out['volume_u3'] = d1 * d2 * d3

    return out

# ── Sanity checks ──────────────────────────────────────────
t1 = parse_dimensions("Product Length: 12 inches")
t2 = parse_dimensions("10 x 8 x 4 inches")
t3 = parse_dimensions("30 cm length")

print(f"'12 inch length'  → explicit_length_u = {t1['explicit_length_u']:.1f}  (expected 1200.0)")
print(f"'10x8x4 inches'   → dim_1_u={t2['dim_1_u']:.1f}  dim_2_u={t2['dim_2_u']:.1f}  (expected 1000, 800)")
print(f"'30 cm length'    → explicit_length_u = {t3['explicit_length_u']:.1f}  (expected {30*39.37:.1f})")
assert t1['explicit_length_u'] == 1200.0, f"Got {t1['explicit_length_u']}"
assert t2['dim_1_u'] == 1000.0, f"Got {t2['dim_1_u']}"
print("✅ parse_dimensions OK — units are PRODUCT_LENGTH units (0.01 inch)")


'12 inch length'  → explicit_length_u = 1200.0  (expected 1200.0)
'10x8x4 inches'   → dim_1_u=1000.0  dim_2_u=800.0  (expected 1000, 800)
'30 cm length'    → explicit_length_u = 1181.1  (expected 1181.1)
✅ parse_dimensions OK — units are PRODUCT_LENGTH units (0.01 inch)


## 4. Load Data & Split

In [5]:
df_train_raw = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
df_test_raw  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

indices = np.arange(len(df_train_raw))
train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=42)

df_train = df_train_raw.iloc[train_idx].reset_index(drop=True)
df_val   = df_train_raw.iloc[val_idx].reset_index(drop=True)
df_test  = df_test_raw.copy()

np.save(os.path.join(OUTPUT_DIR, "train_indices.npy"), train_idx)
np.save(os.path.join(OUTPUT_DIR, "val_indices.npy"),   val_idx)

y_tr = df_train['PRODUCT_LENGTH'].values.astype(float)
print(f"Train: {len(df_train):,}  Val: {len(df_val):,}  Test: {len(df_test):,}")
print(f"PRODUCT_LENGTH: min={y_tr.min():.1f}  p5={np.percentile(y_tr,5):.1f}  median={np.median(y_tr):.1f}  p95={np.percentile(y_tr,95):.1f}  max={y_tr.max():.1f}")


Train: 160,000  Val: 40,000  Test: 50,000
PRODUCT_LENGTH: min=1.0  p5=150.0  median=665.4  p95=3400.0  max=36000000.0


## 5. Extract Features

In [6]:
type_counts  = df_train['PRODUCT_TYPE_ID'].value_counts().to_dict()
type_medians = df_train.groupby('PRODUCT_TYPE_ID')['PRODUCT_LENGTH'].median().to_dict()
type_means   = df_train.groupby('PRODUCT_TYPE_ID')['PRODUCT_LENGTH'].mean().to_dict()
global_median = float(df_train['PRODUCT_LENGTH'].median())
global_mean   = float(df_train['PRODUCT_LENGTH'].mean())

def build_features(df):
    title   = df['TITLE'].fillna('').apply(clean_text)
    bullets = df['BULLET_POINTS'].fillna('').apply(clean_text)
    desc    = df['DESCRIPTION'].fillna('').apply(clean_text)
    combined = (title + ' ' + bullets + ' ' + desc).str.strip()

    dim_df = pd.DataFrame(list(combined.apply(parse_dimensions)))

    feat = dim_df.copy()
    feat['title_len']    = title.str.len()
    feat['bullet_len']   = bullets.str.len()
    feat['desc_len']     = desc.str.len()
    feat['combined_len'] = combined.str.len()
    feat['title_words']  = title.str.split().str.len()
    feat['bullet_words'] = bullets.str.split().str.len()

    pid = df['PRODUCT_TYPE_ID']
    feat['product_type_freq']   = pid.map(type_counts).fillna(0).astype(float)
    feat['product_type_median'] = pid.map(type_medians).fillna(global_median).astype(float)
    feat['product_type_mean']   = pid.map(type_means).fillna(global_mean).astype(float)
    feat['log_type_median']     = np.log1p(feat['product_type_median'])
    feat['PRODUCT_ID']          = df['PRODUCT_ID'].values
    return feat

print("Extracting Train features...")
X_tr = build_features(df_train)
X_tr['PRODUCT_LENGTH'] = df_train['PRODUCT_LENGTH'].values

print("Extracting Val features...")
X_va = build_features(df_val)
X_va['PRODUCT_LENGTH'] = df_val['PRODUCT_LENGTH'].values

print("Extracting Test features...")
X_te = build_features(df_test)

print(f"\nFeature columns: {list(X_tr.columns)}")

has_expl = X_tr['has_explicit_length'].values
expl_u   = X_tr['explicit_length_u'].values
y_tr_arr = X_tr['PRODUCT_LENGTH'].values

mask = (has_expl == 1) & (expl_u > 0)
print(f"\nExplicit length coverage: {mask.sum():,} / {len(X_tr):,} = {mask.mean()*100:.1f}%")
if mask.sum() > 0:
    mape_expl = np.mean(np.abs(y_tr_arr[mask] - expl_u[mask]) / np.maximum(y_tr_arr[mask], 1))
    print(f"Direct extraction MAPE (train, has_explicit): {mape_expl*100:.2f}%")


Extracting Train features...
Extracting Val features...
Extracting Test features...

Feature columns: ['explicit_length_u', 'has_explicit_length', 'dim_1_u', 'dim_2_u', 'dim_3_u', 'max_dim_u', 'min_dim_u', 'mid_dim_u', 'volume_u3', 'measurement_count', 'number_count', 'has_lxw', 'has_lxwxh', 'has_inch', 'has_cm', 'has_mm', 'has_ft', 'title_len', 'bullet_len', 'desc_len', 'combined_len', 'title_words', 'bullet_words', 'product_type_freq', 'product_type_median', 'product_type_mean', 'log_type_median', 'PRODUCT_ID', 'PRODUCT_LENGTH']

Explicit length coverage: 4,786 / 160,000 = 3.0%
Direct extraction MAPE (train, has_explicit): 2816.18%


## 6. Save Parquets

In [7]:
X_tr.to_parquet(os.path.join(OUTPUT_DIR, "X_train_features.parquet"), index=False)
X_va.to_parquet(os.path.join(OUTPUT_DIR, "X_val_features.parquet"),   index=False)
X_te.to_parquet(os.path.join(OUTPUT_DIR, "X_test_features.parquet"),  index=False)
print("✅ Saved. Notebook 02 complete.")
X_tr.head(3)


✅ Saved. Notebook 02 complete.


,explicit_length_u,has_explicit_length,dim_1_u,dim_2_u,dim_3_u,max_dim_u,min_dim_u,mid_dim_u,volume_u3,measurement_count,number_count,has_lxw,has_lxwxh,has_inch,has_cm,has_mm,has_ft,title_len,bullet_len,desc_len,combined_len,title_words,bullet_words,product_type_freq,product_type_median,product_type_mean,log_type_median,PRODUCT_ID,PRODUCT_LENGTH
0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,0,0,0,0,0,0,0,25,0,0,25,6,0,2.0,2027.4825,2027.482500,7.615043,976957,1751.965
1,0.0,0,1300.0,1900.0,0.0,1900.0,1300.0,1900.0,2470000.0,2,7,1,0,1,0,0,0,98,208,176,484,18,31,1115.0,1200.0000,1612.292857,7.090910,1916578,1900.000
2,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0,2,0,0,0,0,0,0,62,0,0,62,10,0,197.0,614.0000,630.687178,6.421622,165335,608.000
